# 案例：MCP 工具发布与外部 MCP 接入

> Last updated: 2026-11-21

## 核心学习目标

本案例以 Chapter-5/6 已实现的酒店搜索工具和领域智能体为基础，说明如何将现有业务工具发布为独立 MCP Server，并使本地 Agent 通过标准协议发现和调用该工具；同时说明如何接入第三方已发布的 MCP，并将其工具绑定给复用自 Chapter-6 的领域智能体。

## 案例设计

案例包含两个相互补充的部分，分别对应工具提供方和工具使用方的视角。

### Part I：发布已有工具

- **目的**：将 Chapter-6 的酒店工具封装为标准 MCP Server，验证工具发现、参数 Schema 和远程调用流程。
- **适用场景**：多个 Agent、应用或部署环境需要复用同一项工具能力，并希望统一维护工具依赖和访问凭据。

### Part II：接入外部 MCP

- **目的**：接入 WeatherAPI 官方 MCP，在不重新实现天气工具的前提下扩展本地 Agent 的能力；WeatherAgent 的角色定义复用自 Chapter-6。
- **适用场景**：需要使用第三方已经发布并维护的 MCP 能力，例如天气、航班或其他领域服务。

## 学习重点

1. 理解本地 LangChain 工具与 MCP 工具在运行边界和调用方式上的差异。
2. 理解 MCP 对工具发现、参数 Schema、调用协议和传输方式的标准化作用。
3. 掌握使用 `FastMCP` 和 `@mcp.tool()` 发布已有工具的方法。
4. 掌握 MCP Client 连接外部 Server、发现工具并绑定给 Agent 的方法。
5. 理解如何复用 Chapter-6 的领域智能体定义，仅替换工具数据源。

### 模块与代码对应关系

| 环节 | 学习内容 | 对应实现 |
|------|----------|----------|
| 酒店工具 | 复用工具实现与领域提示词 | `Chapter-6/domain_agents.py` |
| MCP Server | Streamable HTTP 服务与工具注册 | `Mcp/server.py` |
| MCP 配置 | Server 绑定地址与 Client 访问 URL | `Mcp/settings.py` |
| MCP Client | `/mcp` 端点、工具发现与调用 | `Mcp/client.py` |
| 本地 Agent | 将 MCP Tool 转为 LangChain Tool | Notebook Agent 单元 |
| 外部 MCP | WeatherAPI 官方 `weatherapi-mcp` | Notebook Part II |
| 模型访问 | 统一模型创建接口 | 仓库根目录 `llm.py` |

## 0. 环境准备

本案例使用仓库根目录的共享 `.venv`，模型和地图凭据统一配置在仓库根目录的 `.env` 中。

```powershell
uv pip install -r Chapter-7/Mcp/requirements.txt
```

Part I 的 MCP Server 由 Notebook 单元负责启动和关闭，无需预先执行 `python server.py`。Server 的定义仍保存在 `Mcp/server.py` 中，Notebook 只管理其运行生命周期。

本案例采用 MCP 2025-11-25 规范定义的 Streamable HTTP 传输：

- [Transports](https://modelcontextprotocol.io/specification/2025-11-25/basic/transports)
- [Tools](https://modelcontextprotocol.io/specification/2025-11-25/server/tools)

# Part I：发布已有工具为 MCP Server

本部分说明两个问题：为何在已有本地工具和 Agent 工具调用的基础上仍需引入 MCP；如何将 Chapter-6 的酒店工具发布为独立 MCP Server，并由 Agent 通过标准协议调用。

## 1. 已经具备工具调用能力，为什么仍需要 MCP？

### Chapter-5：工具定义与 Agent 绑定

Chapter-5 将酒店搜索逻辑划分至 `hotel_search.py` 和 `map_api.py`，并在 Notebook 中定义 LangChain 工具：

```python
from hotel_search import search_hotels

@tool
async def recommend_hotel(city, keyword):
    return await search_hotels(city, keyword=keyword)

agent = create_agent(llm, tools=[recommend_hotel])
```

### Chapter-6：领域智能体工具配置

Chapter-6 在 `domain_agents.py` 中重新定义适用于 `HotelAgent` 的工具包装，其内部仍复用 Chapter-5 的底层搜索函数 `search_hotels()`：

```python
@tool
async def recommend_hotel(city, preferences, budget_cny_per_night_max=None):
    result = await search_hotels(city, limit=10, keyword=preferences)
    # 继续执行酒店有效性校验和预算过滤
    return result

"HotelAgent": AgentSpec(
    name="HotelAgent",
    tools=[ToolSpec(tool=recommend_hotel, ...)],
)
```

上述实现已经完成工具定义、Agent 配置和工具调用，并实现了代码层面的模块化。


### MCP 解决的问题

Chapter-5 和 Chapter-6 的共同前提是：Agent 与工具运行在同一个 Python 进程中，工具调用体现为本地函数调用。

当系统仅包含单个 Agent，且工具运行于同一应用进程时，本地 `@tool` 已能满足需求，无需额外引入 MCP。

MCP 面向多 Host、多 Client、多框架和多语言复用工具的场景。依据 MCP 官方文档，MCP 是标准化 AI 应用与数据源、工具之间上下文交换方式的开放协议。Host 管理 MCP Client，Client 与 MCP Server 建立连接；Server 通过标准协议提供工具，Client 使用 `tools/list` 发现工具，并通过 `tools/call` 调用工具。

MCP 的主要优势包括：

1. **统一工具接入接口**：工具通过名称、描述和 JSON Schema 声明能力，Client 无需为每项工具编写独立适配代码。
2. **支持动态发现**：Client 可按需查询 Server 当前提供的工具，无需在 Agent 代码中固化全部工具定义。
3. **支持独立部署与复用**：工具运行于独立 MCP Server，可由多个 Client 使用，工具依赖、凭据和版本由 Server 统一维护。
4. **协议与传输分离**：同一工具协议可通过 stdio 连接本地进程，也可通过 Streamable HTTP 连接远程服务。

MCP 不替代 Agent 的工具调用机制，也不替代工具本身的业务实现。其核心作用是为工具能力提供标准化公开接口，使其能够跨应用、跨进程和跨语言被发现与调用。

## 2. MCP Server 的封装方法

`Mcp/server.py` 负责标准 MCP 注册，承担三项职责：

1. 创建 `FastMCP` Server，并声明服务名称与监听地址。
2. 使用 `@mcp.tool()` 将 Chapter-6 的 `recommend_hotel` 注册为 MCP 工具。
3. 使用标准 Streamable HTTP 传输启动服务。

```python
from domain_agents import recommend_hotel

@mcp.tool() # 根据函数名称、Docstring 和参数类型生成工具元数据
async def recommend_hotel_tool(city, preferences, budget):
    result = await recommend_hotel.ainvoke({
        "city": city,
        "preferences": preferences,
        "budget_cny_per_night_max": budget,
    })
    return result
```

In [1]:
# 配置跨章节导入路径，并为后续代码单元准备运行环境。
from pathlib import Path
import sys

candidates = (
    Path.cwd(),
    Path.cwd() / "Chapter-7",
    Path.cwd().parent,
)
CHAPTER7_DIR = next(
    (
        candidate.resolve()
        for candidate in candidates
        if candidate.name == "Chapter-7"
        and (candidate / "README.md").is_file()
    ),
    None,
)
if CHAPTER7_DIR is None:
    raise RuntimeError("请从仓库根目录或 Chapter-7 启动 Notebook")

MCP_DIR = CHAPTER7_DIR / "Mcp"
for path in (CHAPTER7_DIR, MCP_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from paths import bootstrap_paths
bootstrap_paths()

print("Chapter-7:", CHAPTER7_DIR)

Chapter-7: E:\Book\agent-systems-book\Chapter-7


In [2]:
# 在后台线程启动 Mcp/server.py 中定义的 Streamable HTTP MCP Server。
import atexit
import socket
import threading
import time

import uvicorn

# 若重复执行本单元，先关闭上一次启动的 Server。
if "chapter7_mcp_server" in globals() and chapter7_mcp_server is not None:
    chapter7_mcp_server.should_exit = True
    if "chapter7_mcp_thread" in globals():
        chapter7_mcp_thread.join(timeout=5)

# 1. 通过 settings.py 选择空闲端口并同步 Server/Client 配置。
from settings import configure_server, find_free_port, mcp_url, server_host

chapter7_mcp_host = server_host()
chapter7_mcp_port = find_free_port()
configure_server(port=chapter7_mcp_port)

# 2. 从 server.py 导入唯一的 MCP Server 定义。
from server import mcp

# 3. 生成 Streamable HTTP ASGI 应用，并在后台线程中运行 uvicorn。
chapter7_mcp_app = mcp.streamable_http_app()
chapter7_mcp_server = uvicorn.Server(
    uvicorn.Config(
        chapter7_mcp_app,
        host=chapter7_mcp_host,
        port=chapter7_mcp_port,
        log_level="warning",
        access_log=False,
    )
)
chapter7_mcp_thread = threading.Thread(
    target=chapter7_mcp_server.run,
    daemon=True,
)

def _stop_chapter7_mcp() -> None:
    chapter7_mcp_server.should_exit = True

atexit.register(_stop_chapter7_mcp)
chapter7_mcp_thread.start()

# 4. 等待端口就绪后再供后续单元调用。
for _ in range(50):
    try:
        with socket.create_connection(
            (chapter7_mcp_host, chapter7_mcp_port),
            timeout=0.2,
        ):
            break
    except OSError:
        time.sleep(0.1)
else:
    raise RuntimeError("MCP Server 启动超时")

print("MCP Server:", mcp_url())

INFO:mcp.server.streamable_http_manager:StreamableHTTP session manager started


MCP Server: http://127.0.0.1:63401/mcp


In [3]:
# 读取 server.py 中的实际 MCP Server 与工具注册代码，检验封装内容。
import inspect
from server import mcp, recommend_hotel_tool

print("MCP server name:", mcp.name)
print(inspect.getsource(recommend_hotel_tool))

MCP server name: hotel-recommendation
@mcp.tool()
async def recommend_hotel_tool(
    city: str,
    preferences: Optional[str] = None,
    budget_cny_per_night_max: Optional[int] = None,
) -> Dict[str, Any]:
    """Expose the Chapter-6 hotel tool as an MCP tool.

    MCP 2025-11-25 requires tool metadata such as name, description and
    ``inputSchema``. ``@mcp.tool`` derives that metadata from this function.

    Reference:
      https://modelcontextprotocol.io/specification/2025-11-25/server/tools
    """
    result = await recommend_hotel.ainvoke(
        {
            "city": city,
            "preferences": preferences,
            "budget_cny_per_night_max": budget_cny_per_night_max,
        }
    )
    return dict(result) if isinstance(result, dict) else {"result": result}



## 3. MCP Client 的工具发现与调用

MCP Client 通过工具发现接口读取 Server 提供的工具名称、描述和 JSON Schema，并将远程工具转换为 LangChain Tool。

本案例应仅发现 `recommend_hotel_tool` 一项工具。

In [4]:
# 通过 Streamable HTTP 连接 MCP Server，发现工具名称、描述和参数 Schema。
from settings import mcp_url
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "hotel": {
            "transport": "streamable_http",
            "url": mcp_url(),
            "timeout": 10.0,
            "sse_read_timeout": 300.0,
        }
    }
)

mcp_tools = await mcp_client.get_tools()
print("Discovered tools:", [tool.name for tool in mcp_tools])
print("Tool schema:", mcp_tools[0].args_schema)

INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.client.streamable_http:Negotiated protocol version: 2025-11-25


INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest


INFO:mcp.server.streamable_http:Terminating session: None


Discovered tools: ['recommend_hotel_tool']
Tool schema: {'properties': {'city': {'title': 'City', 'type': 'string'}, 'preferences': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Preferences'}, 'budget_cny_per_night_max': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'Budget Cny Per Night Max'}}, 'required': ['city'], 'title': 'recommend_hotel_toolArguments', 'type': 'object'}


### 工具直接调用验证

先由程序直接调用 MCP 工具，以验证协议连接和工具返回是否正常。响应应为结构化酒店候选数据，而非最终推荐文本。

In [5]:
# 在接入大模型前，由程序直接调用 MCP 工具，验证协议链路和结构化结果。
from client import fetch_hotels_via_mcp

hotel_result = await fetch_hotels_via_mcp("大同", preferences="近古城")
assert hotel_result is not None

print("Search query:", hotel_result.get("search_query"))
print("Candidate count:", hotel_result.get("count"))
for index, hotel in enumerate(hotel_result.get("hotels", [])[:5], 1):
    print(
        f"{index}. {hotel.get('name')} | "
        f"{hotel.get('district') or hotel.get('address')}"
    )

INFO:mcp.client.streamable_http:Negotiated protocol version: 2025-11-25


INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.server.lowlevel.server:Processing request of type CallToolRequest


INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest


INFO:mcp.server.streamable_http:Terminating session: None


Search query: 近古城 酒店
Candidate count: 10
1. 大同古城 | 平城区
2. 汉庭酒店(大同古城北门店) | 平城区
3. 高渡精品酒店(大同古城华严寺店) | 平城区
4. 如家酒店·neo(大同古城迎宾街店) | 平城区
5. 雲都酒店(大同古城店) | 平城区


## 4. Agent 使用 MCP 工具

将远程 MCP Tool 绑定至本地 Agent。模型负责判断调用时机和参数，并依据工具返回的酒店候选生成最终推荐。

该场景中跨进程调用的是工具能力，Agent 仍运行于当前 Notebook 进程。

In [6]:
# 将发现的远程 MCP Tool 绑定到本地 LangChain Agent。
# Shared model factory from repository-root llm.py.
from llm import create_llm as create_shared_llm
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

agent_tools = [
    tool for tool in mcp_tools
    if tool.name == "recommend_hotel_tool"
]
agent = create_agent(
    create_shared_llm(),
    tools=agent_tools,
    system_prompt=(
        "你是酒店推荐助手。只能通过 recommend_hotel_tool 查询候选酒店，"
        "再根据返回字段推荐一家酒店。"
    ),
    checkpointer=MemorySaver(),
)
print("Agent tools:", [tool.name for tool in agent_tools])

Agent tools: ['recommend_hotel_tool']


In [7]:
# Agent 执行完成后，一次性输出完整的工具结果和最终回答。
import json

USER_QUERY = "推荐一家大同古城附近、预算不超过500元的酒店"

def _full_text(value):
    """将工具返回内容或模型消息转换为完整文本。"""
    content = getattr(value, "content", value)
    if isinstance(content, str):
        try:
            content = json.loads(content)
        except json.JSONDecodeError:
            return content
    if isinstance(content, (dict, list)):
        return json.dumps(content, ensure_ascii=False, indent=2, default=str)
    return str(content)

async for event in agent.astream_events(
    {"messages": [("user", USER_QUERY)]},
    {"configurable": {"thread_id": "chapter7_mcp_notebook"}},
    version="v2",
):
    kind = event["event"]

    if kind == "on_tool_start":
        print("MCP tool call:", event.get("name"))
        print("Arguments:", _full_text(event["data"].get("input")))

    elif kind == "on_tool_end":
        print("Tool returned:")
        print(_full_text(event["data"].get("output")))

    elif kind == "on_chain_end" and event.get("name") == "LangGraph":
        output = event.get("data", {}).get("output") or {}
        messages = output.get("messages", []) if isinstance(output, dict) else []
        if messages:
            print("\nFinal answer:")
            print(_full_text(messages[-1]))
        break

INFO:mcp.client.streamable_http:Negotiated protocol version: 2025-11-25


INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.server.lowlevel.server:Processing request of type CallToolRequest


MCP tool call: recommend_hotel_tool
Arguments: {
  "city": "大同",
  "budget_cny_per_night_max": 500,
  "preferences": "大同古城附近"
}


INFO:mcp.server.streamable_http:Terminating session: None


INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest


INFO:mcp.server.streamable_http:Terminating session: None


Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"city\": \"大同\",\n  \"search_query\": \"大同古城附近 酒店\",\n  \"preferences\": \"大同古城附近\",\n  \"budget_cny_per_night_max\": 500,\n  \"hotels\": [\n    {\n      \"name\": \"钰雅轩民宿·中式合院别墅(大同古城代王府店)\",\n      \"district\": \"平城区\",\n      \"address\": \"武定街铂蓝五龙巷27院2号\",\n      \"tel\": \"13817686466,15635229520,18935224510\",\n      \"location\": \"113.307117,40.107581\",\n      \"rating\": 5.0,\n      \"avg_price_cny\": null,\n      \"type\": \"hotel\"\n    },\n    {\n      \"name\": \"恒瑞精品民宿(大同古城墙店)\",\n      \"district\": \"平城区\",\n      \"address\": \"大同市福华园西区20号楼3单元102\",\n      \"tel\": \"13935233892\",\n      \"location\": \"113.29953,40.09576\",\n      \"rating\": 5.0,\n      \"avg_price_cny\": null,\n      \"type\": \"hotel\"\n    },\n    {\n      \"name\": \"云树酒店(大同古城华严寺店)\",\n      \"district\": \"平城区\",\n      \"address\": \"山西省大同市平城区武定北路26号院内共创空间\",\n      \"tel\": \"15698392204\",\n      \"location\": \"113.310953,40.11105


Final answer:
根据查询结果，我为您推荐以下酒店：

**推荐酒店：钰雅轩民宿·中式合院别墅(大同古城代王府店)**

- **位置**：平城区武定街铂蓝五龙巷27院2号，位于大同古城内，靠近代王府
- **评分**：5.0分（满分）
- **特点**：
  - 中式合院别墅风格，具有浓厚的传统文化氛围
  - 地理位置优越，位于古城核心区域，方便游览大同古城墙、华严寺等景点
  - 评分最高，说明住客体验非常好
  - 民宿形式，能提供更个性化的服务和更舒适的居住体验

这家酒店符合您的预算要求（不超过500元/晚），且位于大同古城附近，交通便利，是游览大同古城的理想选择。


In [8]:
# Part I 完成后关闭 Notebook 启动的 MCP Server。
if "chapter7_mcp_server" in globals() and chapter7_mcp_server is not None:
    chapter7_mcp_server.should_exit = True
    if "chapter7_mcp_thread" in globals():
        chapter7_mcp_thread.join(timeout=10)

    stopped = not chapter7_mcp_thread.is_alive()
    print("MCP Server stopped:", stopped)
else:
    print("MCP Server was not started by this Notebook.")

INFO:mcp.server.streamable_http_manager:StreamableHTTP session manager shutting down


MCP Server stopped: True


# Part II：接入外部 MCP

Part I 说明了如何发布自有工具；本部分说明如何接入第三方已经发布的 MCP，并将其工具绑定给本地 Agent。

## 接入 weatherapi-mcp
WeatherAPI.com 于 2026 年 3 月 22 日发布了 `weatherapi-mcp 1.0.0`。该 MCP 提供实时天气、天气预报、历史天气、海洋天气、天文信息、空气质量等能力。借助这一已经发布的外部 MCP，本地 Agent 无需自行实现天气 API 适配，即可扩展天气查询能力。

WeatherAPI 官方 MCP 信息如下：

- 包名：`weatherapi-mcp`
- 作者：`WeatherAPI.com`
- 仓库：https://github.com/weatherapicom/weatherapi-mcp
- License：MIT
- 运行命令：`npx -y weatherapi-mcp`
- 认证环境变量：`WEATHERAPI_KEY`

该 Server 提供 `get_current_weather`、`get_forecast`、`get_history`、`search_locations` 等工具。本案例采用 stdio 传输，由本地进程启动并连接外部 MCP Server。

本案例不重新定义 WeatherAgent，而是复用 Chapter-6 的 `DOMAIN_AGENT_SPECS["WeatherAgent"]`。角色名称、职责说明和 System Prompt 均沿用 Chapter-6，仅将 `tools=` 的来源替换为 WeatherAPI MCP 工具。

In [9]:
# 本单元接入 WeatherAPI 官方 MCP，并复用 Chapter-6 WeatherAgent。
import os
import shutil

from external_mcp import run_agent_once, stdio_mcp_tools

# Cross-chapter import from Chapter-6/domain_agents.py.
from domain_agents import DOMAIN_AGENT_SPECS as CHAPTER6_AGENT_SPECS
# Shared model factory from repository-root llm.py.
from llm import create_llm as create_shared_llm
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# 1. 读取认证信息，并检查 npx 是否可用。
WEATHERAPI_KEY = (os.getenv("WEATHERAPI_KEY") or "").strip()
WEATHER_MCP_READY = bool(WEATHERAPI_KEY) and not WEATHERAPI_KEY.lower().startswith("your-")
npx = shutil.which("npx") or shutil.which("npx.cmd")
if not npx:
    raise RuntimeError("未找到 npx，请安装 Node.js 后重试")

# 2. 配置外部 MCP 的 stdio 启动参数和认证环境变量。
weather_env = {**os.environ, "WEATHERAPI_KEY": WEATHERAPI_KEY}

# 3. 启动外部 MCP，并发现 WeatherAPI 提供的全部工具。
async with stdio_mcp_tools(
    npx,
    ["-y", "weatherapi-mcp"],
    weather_env,
) as weather_mcp_tools:
    print("WeatherAPI key configured:", WEATHER_MCP_READY)
    print("External MCP tools:", [tool.name for tool in weather_mcp_tools])

    # 4. 复用 Chapter-6 WeatherAgent 的角色定义与 System Prompt。
    weather_spec = CHAPTER6_AGENT_SPECS["WeatherAgent"]
    print("Reused Chapter-6 WeatherAgent:", weather_spec.name)
    print("Role:", weather_spec.description)

    # 5. 仅将工具来源替换为外部 WeatherAPI MCP。
    weather_agent = create_agent(
        create_shared_llm(),
        tools=weather_mcp_tools,
        system_prompt=weather_spec.system_prompt,
        checkpointer=MemorySaver(),
    )

    if not WEATHER_MCP_READY:
        print(
            "跳过真实天气调用：请在仓库根目录 .env 中配置有效的 "
            "WEATHERAPI_KEY。"
        )
    else:
        # 6. 执行 Agent，并完整、一次性输出工具结果和最终回答。
        try:
            await run_agent_once(
                weather_agent,
                "北京今天天气怎么样？",
                "chapter7_external_weather",
            )
        except Exception as exc:
            print(f"WeatherAPI MCP 调用失败: {exc}")

WeatherAPI key configured: True
External MCP tools: ['get_current_weather', 'get_forecast', 'get_history', 'get_future_weather', 'get_marine_weather', 'get_astronomy', 'get_timezone', 'search_locations', 'ip_lookup', 'get_alerts', 'get_sports']
Reused Chapter-6 WeatherAgent: WeatherAgent
Role: 根据城市和确切日期查询天气预报；每次请求只查询一个城市、一个日期的天气。


MCP tool call: get_current_weather
Arguments: {
  "q": "北京"
}


Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"location\": {\n    \"name\": \"北京\",\n    \"region\": \"Beijing\",\n    \"country\": \"中国\",\n    \"lat\": 39.9289,\n    \"lon\": 116.3883,\n    \"tz_id\": \"Asia/Shanghai\",\n    \"localtime_epoch\": 1795274179,\n    \"localtime\": \"2026-11-21 23:16\"\n  },\n  \"current\": {\n    \"last_updated_epoch\": 1795273200,\n    \"last_updated\": \"2026-11-21 23:00\",\n    \"temp_c\": 1,\n    \"temp_f\": 33.7,\n    \"is_day\": 0,\n    \"condition\": {\n      \"text\": \"Smoky haze\",\n      \"icon\": \"//cdn.weatherapi.com/weather/64x64/night/149.png\",\n      \"code\": 1036\n    },\n    \"wind_mph\": 4,\n    \"wind_kph\": 6.5,\n    \"wind_degree\": 204,\n    \"wind_dir\": \"SSW\",\n    \"pressure_mb\": 1030,\n    \"pressure_in\": 30.41,\n    \"precip_mm\": 0,\n    \"precip_in\": 0,\n    \"humidity\": 51,\n    \"cloud\": 0,\n    \"feelslike_c\": -1.1,\n    \"feelslike_f\": 30,\n    \"windchill_c\": -1.1,\n    \"windchill_f\": 30,\n 

\nFinal answer:
根据查询结果，北京今天（2026-11-21）的天气情况如下：

**当前天气状况：**
- **气温**：1°C（体感温度 -1.1°C）
- **天气现象**：烟雾/霾（Smoky haze）
- **风向风速**：西南偏南风，6.5公里/小时
- **湿度**：51%
- **气压**：1030百帕
- **能见度**：6.5公里
- **降水概率**：降雨2%，降雪8%
- **云量**：0%（晴朗无云）

目前北京夜间气温较低，有烟雾或霾的情况，能见度一般。建议外出时注意保暖并佩戴口罩。


## 接入百度地图 MCP

百度地图 MCP 是百度地图官方提供的 LBS MCP Server，支持本地 stdio、SSE 和远程 Streamable HTTP。本案例使用 npm 包的本地 stdio 方式。

- 仓库：<https://github.com/baidu-maps/mcp>
- npm 包：`@baidumap/mcp-server-baidu-map`
- License：MIT
- 申请 Key：<https://lbsyun.baidu.com/apiconsole/key>
- 启动命令：`npx -y @baidumap/mcp-server-baidu-map`
- 官方 Node 环境变量：`BAIDU_MAP_API_KEY`

In [10]:
# 接入百度地图 MCP，发现工具后交由 Chapter-6 HotelAgent 自主选择。
import os
import shutil

from external_mcp import run_agent_once, stdio_mcp_tools

# Cross-chapter import from Chapter-6/domain_agents.py.
from domain_agents import DOMAIN_AGENT_SPECS as CHAPTER6_AGENT_SPECS
# Shared model factory from repository-root llm.py.
from llm import create_llm as create_shared_llm
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# 1. 读取百度地图 Key。仓库已配置 BAIDU_MAP_AK 时可直接复用。
BAIDU_MAP_KEY = (
    os.getenv("BAIDU_MAPS_API_KEY")
    or os.getenv("BAIDU_MAP_API_KEY")
    or os.getenv("BAIDU_MAP_AK")
    or ""
).strip()
BAIDU_MCP_READY = bool(BAIDU_MAP_KEY)
npx = shutil.which("npx") or shutil.which("npx.cmd")
if not npx:
    raise RuntimeError("未找到 npx，请安装 Node.js 后重试")

# 2. 将已有 BAIDU_MAP_AK 映射为官方 npm 包使用的 BAIDU_MAP_API_KEY。
baidu_env = {**os.environ, "BAIDU_MAP_API_KEY": BAIDU_MAP_KEY}

# 3. 启动百度地图 MCP，并调用 tools/list 发现全部工具。
async with stdio_mcp_tools(
    npx,
    ["-y", "@baidumap/mcp-server-baidu-map"],
    baidu_env,
) as baidu_tools:
    print("Baidu Map MCP tools:", [tool.name for tool in baidu_tools])

    # 4. 复用 Chapter-6 HotelAgent 的角色定义与 System Prompt。
    hotel_spec = CHAPTER6_AGENT_SPECS["HotelAgent"]
    baidu_hotel_agent = create_agent(
        create_shared_llm(),
        tools=baidu_tools,
        system_prompt=hotel_spec.system_prompt,
        checkpointer=MemorySaver(),
    )
    print("Reused Chapter-6 HotelAgent:", hotel_spec.name)
    print("Agent-visible tools:", [tool.name for tool in baidu_tools])

    if not BAIDU_MCP_READY:
        print("跳过真实查询：请配置 BAIDU_MAP_API_KEY 或 BAIDU_MAP_AK。")
    else:
        # 5. 由 Agent 根据工具描述自主选择工具，不接受预先指定。
        try:
            await run_agent_once(
                baidu_hotel_agent,
                "查询大同古城附近评分较高的酒店",
                "chapter7_baidu_mcp",
            )
        except Exception as exc:
            print(f"百度地图 MCP 调用失败: {exc}")

Baidu Map MCP tools: ['map_geocode', 'map_reverse_geocode', 'map_search_places', 'map_place_details', 'map_distance_matrix', 'map_directions', 'map_weather', 'map_ip_location', 'map_road_traffic', 'map_poi_extract']
Reused Chapter-6 HotelAgent: HotelAgent
Agent-visible tools: ['map_geocode', 'map_reverse_geocode', 'map_search_places', 'map_place_details', 'map_distance_matrix', 'map_directions', 'map_weather', 'map_ip_location', 'map_road_traffic', 'map_poi_extract']


MCP tool call: map_search_places
Arguments: {
  "query": "酒店",
  "region": "大同"
}


Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"result_type\": \"poi_type\",\n  \"query_type\": \"general\",\n  \"results\": [\n    {\n      \"name\": \"海驿酒店(大同云冈店)\",\n      \"location\": {\n        \"lat\": 40.038893,\n        \"lng\": 113.179085\n      },\n      \"address\": \"同泉路2846号\",\n      \"province\": \"山西省\",\n      \"city\": \"大同市\",\n      \"area\": \"云冈区\",\n      \"street_id\": \"664140a847001f8e441a32f2\",\n      \"telephone\": \"15635216329\",\n      \"detail\": 1,\n      \"uid\": \"50686990636706b838e4a6c9\"\n    },\n    {\n      \"name\": \"如家华驿精选酒店(大同云冈店)\",\n      \"location\": {\n        \"lat\": 40.033134,\n        \"lng\": 113.171356\n      },\n      \"address\": \"郝家寺世纪阳光小区外围甲3号楼\",\n      \"province\": \"山西省\",\n      \"city\": \"大同市\",\n      \"area\": \"云冈区\",\n      \"telephone\": \"03527010088\",\n      \"detail\": 1,\n      \"uid\": \"5903bf80205d4d159ff77990\"\n    },\n    {\n      \"name\": \"帕罗智选酒店(大同南站机场方特店)\",\n      \"location\": {\n  

MCP tool call: map_geocode
Arguments: {
  "address": "大同古城"
}
Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"location\": {\n    \"lng\": 113.30868301049725,\n    \"lat\": 40.09977880014242\n  },\n  \"precise\": 0,\n  \"confidence\": 50,\n  \"comprehension\": 100,\n  \"level\": \"旅游景点\"\n}",
    "id": "lc_601f8125-8195-4c20-b7fa-9428d19baa61"
  }
]


MCP tool call: map_search_places
Arguments: {
  "query": "酒店",
  "location": "40.099779,113.308683"
}
Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"result_type\": \"poi_type\",\n  \"query_type\": \"general\",\n  \"results\": [\n    {\n      \"name\": \"迦南宾馆(大同古城钟楼店)\",\n      \"location\": {\n        \"lat\": 40.099508,\n        \"lng\": 113.307237\n      },\n      \"address\": \"清远街B区11号\",\n      \"province\": \"山西省\",\n      \"city\": \"大同市\",\n      \"area\": \"平城区\",\n      \"street_id\": \"f441b8271f4886aa8bfc060e\",\n      \"telephone\": \"(0352)2838098\",\n      \"detail\": 1,\n      \"uid\": \"f441b8271f4886aa8bfc060e\"\n    },\n    {\n      \"name\": \"高渡精品酒店(大同古城华严寺店)\",\n      \"location\": {\n        \"lat\": 40.10234,\n        \"lng\": 113.308535\n      },\n      \"address\": \"武定街83号\",\n      \"province\": \"山西省\",\n      \"city\": \"大同市\",\n      \"area\": \"平城区\",\n      \"street_id\": \"dad772f671f0bbf0fbf0cbb3\",\n      \"telephone\": \"03528885511,03

MCP tool call: map_place_details
Arguments: {
  "uid": "f441b8271f4886aa8bfc060e",
  "scope": "2"
}
MCP tool call: map_place_details
Arguments: {
  "uid": "dad772f671f0bbf0fbf0cbb3",
  "scope": "2"
}
MCP tool call: map_place_details
Arguments: {
  "uid": "65c9c32094333443b45f1ea4",
  "scope": "2"
}
MCP tool call: map_place_details
Arguments: {
  "uid": "7944aef168d28bd204fef529",
  "scope": "2"
}
MCP tool call: map_place_details
Arguments: {
  "uid": "98b9b8d27771fbb22fb17e57",
  "scope": "2"
}
Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"uid\": \"dad772f671f0bbf0fbf0cbb3\",\n  \"name\": \"高渡精品酒店(大同古城华严寺店)\",\n  \"location\": {\n    \"lat\": 40.10234,\n    \"lng\": 113.308535\n  },\n  \"address\": \"武定街83号\",\n  \"province\": \"山西省\",\n  \"city\": \"大同市\",\n  \"area\": \"平城区\",\n  \"street_id\": \"dad772f671f0bbf0fbf0cbb3\",\n  \"detail\": 1,\n  \"detail_info\": {\n    \"classified_poi_tag\": \"酒店;舒适型\",\n    \"tag\": \"酒店\",\n    \"navi_location\": {\n      \"lat\": 40

... (2 outputs left out)

Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"uid\": \"2c00002c27df90e80a69d9f9\",\n  \"name\": \"古城里酒店(大同古城华严寺店)\",\n  \"location\": {\n    \"lat\": 40.095729,\n    \"lng\": 113.304797\n  },\n  \"address\": \"欢乐街金华苑小区4幢4单元\",\n  \"province\": \"山西省\",\n  \"city\": \"大同市\",\n  \"area\": \"平城区\",\n  \"detail\": 1,\n  \"detail_info\": {\n    \"classified_poi_tag\": \"酒店;经济型\",\n    \"tag\": \"酒店\",\n    \"navi_location\": {\n      \"lat\": 40.095733,\n      \"lng\": 113.304707\n    },\n    \"type\": \"hotel\",\n    \"detail_url\": \"http://api.map.baidu.com/place/detail?output=html&source=placeapi_v3&uid=2c00002c27df90e80a69d9f9\",\n    \"overall_rating\": \"4.6\",\n    \"service_rating\": \"0\",\n    \"facility_rating\": \"0\",\n    \"hygiene_rating\": \"0\",\n    \"image_num\": \"101\",\n    \"comment_num\": \"0\",\n    \"favorite_num\": \"0\",\n    \"checkin_num\": \"0\",\n    \"level\": \"舒适型\",\n    \"shop_hours\": \"\",\n    \"parent_id\": \"de18213250a43cc6e507381e\

MCP tool call: map_place_details
Arguments: {
  "uid": "f441b8271f4886aa8bfc060e",
  "scope": "2"
}


Tool returned:
[
  {
    "type": "text",
    "text": "{\n  \"uid\": \"f441b8271f4886aa8bfc060e\",\n  \"name\": \"迦南宾馆(大同古城钟楼店)\",\n  \"location\": {\n    \"lat\": 40.099508,\n    \"lng\": 113.307237\n  },\n  \"address\": \"清远街B区11号\",\n  \"province\": \"山西省\",\n  \"city\": \"大同市\",\n  \"area\": \"平城区\",\n  \"street_id\": \"f441b8271f4886aa8bfc060e\",\n  \"detail\": 1,\n  \"detail_info\": {\n    \"classified_poi_tag\": \"酒店;经济型\",\n    \"tag\": \"酒店\",\n    \"navi_location\": {\n      \"lat\": 40.09958,\n      \"lng\": 113.307252\n    },\n    \"type\": \"hotel\",\n    \"detail_url\": \"http://api.map.baidu.com/place/detail?output=html&source=placeapi_v3&uid=f441b8271f4886aa8bfc060e\",\n    \"overall_rating\": \"4.3\",\n    \"service_rating\": \"4.4\",\n    \"facility_rating\": \"4.4\",\n    \"hygiene_rating\": \"4.4\",\n    \"image_num\": \"115\",\n    \"comment_num\": \"18\",\n    \"favorite_num\": \"0\",\n    \"checkin_num\": \"0\",\n    \"level\": \"经济型\",\n    \"environment_exterior

\nFinal answer:
根据我获取的信息，工具返回的酒店详情中不包含参考价格信息。让我基于已有的评分、地址、类型等信息，推荐大同古城附近评分较高的酒店。

以下是大同古城附近评分较高的5家酒店推荐：

---

**1. 琵琶老店客栈(大同古城墙华严寺店)**
- 地址：华严寺广场东56米
- 评分：4.7分
- 类型：舒适型
- 推荐理由：评分高达4.7分，服务、设施、卫生评分均为4.8分，位于华严寺广场附近，地理位置优越。

**2. 唐坊·涵云馆(大同古城华严寺店)**
- 地址：清远街113号潘家园B区外侧
- 评分：4.7分
- 类型：舒适型
- 推荐理由：评分4.7分，服务、设施、卫生评分均达4.9分，提供咖啡厅、洗衣服务等完善设施。

**3. 湘禾家庭酒店(大同古城店)**
- 地址：教场街1号华林商厦5楼
- 评分：4.7分
- 类型：经济型
- 推荐理由：评分4.7分，经济型酒店中评分较高，位于华林商厦内，交通便利。

**4. 高渡精品酒店(大同古城华严寺店)**
- 地址：武定街83号
- 评分：4.6分
- 类型：舒适型
- 推荐理由：评分4.6分，各项评分均衡，提供会议厅、商务中心、餐厅等完善设施。

**5. 颐原居(大同古城店)**
- 地址：鼓楼东街24号院
- 评分：4.6分
- 类型：舒适型
- 推荐理由：评分4.6分，位于鼓楼东街，靠近古城核心区域，方便游览景点。

---

注：工具未提供参考价格信息，以上推荐基于评分、地址位置和酒店类型。如需了解具体价格，建议直接联系酒店或通过预订平台查询。


## 扩展练习：为 Chapter-6 FlightAgent 接入飞常准 MCP

请模仿本案例接入 WeatherAPI MCP 的方式，为 Chapter-6 定义的 `FlightAgent` 配置飞常准 Aviation MCP。保持 Chapter-6 的角色名称、职责说明和 System Prompt，仅将 `tools=` 的来源替换为飞常准 MCP 工具。

### 飞常准 Aviation MCP 能力

飞常准 Aviation MCP 面向航空出行场景，提供 9 个工具，主要能力包括：

- 按出发地、目的地和日期查询直飞航班。
- 按航班号和日期查询指定航班。
- 查询联程航班方案。
- 查询航班舒适度、准点率、座位、餐食和行李额度。
- 根据飞机注册号查询实时位置。
- 查询机场未来天气。
- 查询两地间在售航班的行程摘要、最低票价和最短时长。
- 查询结构化票价与舱位信息。

### 接入信息

- 官方文档：<https://ai.variflight.com/docs/aviation>
- 申请 API Key：<https://ai.variflight.com/keys>
- Streamable HTTP 端点：`https://ai.variflight.com/servers/aviation/mcp`
- 认证请求头：`X-API-Key`
- 建议环境变量：

```text
VARIFLIGHT_MCP_URL=https://ai.variflight.com/servers/aviation/mcp
VARIFLIGHT_API_KEY=your_variflight_api_key
```

### 配置示意

```python
{
    "variflight": {
        "transport": "streamable_http",
        "url": os.getenv(
            "VARIFLIGHT_MCP_URL",
            "https://ai.variflight.com/servers/aviation/mcp",
        ),
        "headers": {
            "X-API-Key": os.getenv("VARIFLIGHT_API_KEY", ""),
        },
    }
}
```

### 练习步骤

1. 使用上述配置创建 `MultiServerMCPClient`。
2. 通过 `get_tools()` 发现飞常准 MCP 工具，并打印工具名称与参数 Schema。
3. 从 Chapter-6 读取 `DOMAIN_AGENT_SPECS["FlightAgent"]`。
4. 使用飞常准 MCP 工具和 Chapter-6 FlightAgent 的 System Prompt 创建 Agent。
5. 使用例如“查询 2026-12-01 上海到北京的可售航班”作为测试任务。
6. 比较替换前本地航班工具与替换后外部 MCP 的数据字段和能力范围。

## 小结

本案例覆盖三类 MCP 使用方式：

| 场景 | 工具来源 | Agent 获取工具的方式 |
|---|---|---|
| 发布自有工具 | Chapter-6 `recommend_hotel` | 连接本项目 MCP Server，通过 `tools/list` 发现 `recommend_hotel_tool` |
| 接入天气 MCP | WeatherAPI 官方 `weatherapi-mcp` | 启动外部 stdio Server，发现天气工具并复用 Chapter-6 WeatherAgent |
| 接入地图 MCP | 百度地图官方 `@baidumap/mcp-server-baidu-map` | 发现全部地图工具，由复用自 Chapter-6 的 HotelAgent 自主选择 |

三类案例共同说明了 MCP 的核心作用：

1. 将已有工具通过标准协议发布为可复用能力。
2. 支持 Client 动态发现工具名称、描述和参数 Schema。
3. 允许同一 Agent 兼容本项目工具和第三方 MCP 工具，而无需针对每项服务重写调用逻辑。

> MCP 不替代 Agent 或工具业务逻辑，而是标准化工具能力的公开、发现和调用方式，使其能够跨进程、跨应用和跨语言复用。